[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/07_%E6%8A%BD%E5%87%BA%E6%B3%95%E3%83%BB%E5%AE%9F%E9%A8%93%E8%A8%88%E7%94%BB%E3%83%BB%E6%A0%BC%E5%B7%AE%E6%8C%87%E6%A8%99.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。そのまま上から順に実行できます。

# 統計2級-07. 標本抽出法・実験計画・歪度尖度・ジニ係数

**🎯 できるようになること**
- 各種の標本抽出法とフィッシャーの3原則を説明できる
- 歪度・尖度で分布の形を表せる
- ローレンツ曲線・ジニ係数で格差を測れる

**前提**：統計3級04・05　/　**所要**：約30分　/　**レベル**：統計検定2級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
rng = np.random.default_rng(0)   # 乱数生成器（seedで結果を固定）

## 1. 標本抽出法

母集団から標本を取る代表的な方法。**偏りを防ぐ**のが目的です。

| 方法 | やり方 | 特徴 |
|---|---|---|
| 単純無作為抽出 | 全員から完全にランダム | 基本だが手間 |
| 系統抽出 | 名簿から一定間隔で抽出 | 簡単／周期に注意 |
| 層化抽出 | 層（年代等）に分け各層から抽出 | 各層を確実に代表 |
| クラスター抽出 | 集団(学校等)ごと抽出し全数調査 | コスト減／誤差増 |
| 多段抽出 | 段階的に抽出（県→市→人） | 大規模調査で実用的 |

In [ ]:
rng = np.random.default_rng(1)             # 乱数生成器（seed固定）
# 1000人の母集団（idと年代）を作る
pop = pd.DataFrame({'id': range(1, 1001),
                    '年代': rng.choice(['10代','20代','30代','40代'], 1000)})

# 単純無作為抽出
simple = pop.sample(n=40, random_state=1)            # 完全ランダムに40人
# 系統抽出（25人おき）
systematic = pop.iloc[rng.integers(0,25)::25]        # 一定間隔で抽出
# 層化抽出（各年代から10人ずつ）
strat = pop.groupby('年代', group_keys=False).sample(n=10, random_state=1)  # 層ごとに抽出
print('単純無作為:', len(simple), '人')
print('系統抽出  :', len(systematic), '人')
print('層化抽出の年代構成:\n', strat['年代'].value_counts())

## 2. フィッシャーの3原則（実験計画）

実験で正しく効果を測るための3原則。

1. **反復**：同じ条件を複数回 → 偶然のばらつきを評価できる
2. **無作為化（ランダム化）**：処理の割り当てをランダムに → 偏りを防ぐ
3. **局所管理（ブロック化）**：似た条件をブロックにまとめて比較 → 誤差を小さく

> これらは一元配置分散分析（`04_統計_2級/03`）の前提になっています。

## 3. 歪度（わいど）と尖度（せんど）

分布の「形」を数で表す指標。
- **歪度**：左右の非対称さ。正なら右に裾が長い、負なら左に裾
- **尖度**：尖り具合・裾の重さ（正規分布を基準0とする定義）

In [ ]:
normal_data = rng.normal(0, 1, 5000)       # 左右対称な正規データ
right_skew = rng.exponential(1, 5000)   # 右に裾が長い
# それぞれの歪度（左右の非対称さ）と尖度（とがり具合）を計算
for name, d in [('正規っぽい', normal_data), ('右に裾(指数)', right_skew)]:
    print(f'{name}: 歪度={stats.skew(d):.2f}, 尖度={stats.kurtosis(d):.2f}')

plt.hist(right_skew, bins=40); plt.title('右に裾が長い分布（歪度>0）'); plt.show()

## 4. ローレンツ曲線とジニ係数

所得などの**格差・不平等**を測る指標。
- ローレンツ曲線：人を所得の低い順に並べ、累積人口比と累積所得比を描く
- ジニ係数：完全平等線とローレンツ曲線で囲む面積の2倍。0(平等)〜1(独占)

In [ ]:
# ジニ係数とローレンツ曲線を計算する関数
def gini(x):
    x = np.sort(np.asarray(x, float))      # 小さい順に並べる
    n = len(x)
    cum = np.cumsum(x) / x.sum()            # 累積所得の割合
    lorenz = np.insert(cum, 0, 0)           # 先頭に0を足す（ローレンツ曲線）
    pop = np.linspace(0, 1, n + 1)          # 累積人口の割合
    area = np.sum(np.diff(pop) * (lorenz[:-1] + lorenz[1:]) / 2)  # 台形則で曲線下の面積
    g = 1 - 2 * area                        # ジニ係数
    return g, pop, lorenz

incomes = rng.lognormal(3, 0.6, 500)   # 所得（不平等あり）
g, popshare, lorenz = gini(incomes)        # ジニ係数とローレンツ曲線を求める
plt.plot(popshare, lorenz, label='ローレンツ曲線')
plt.plot([0,1],[0,1],'--', label='完全平等線')  # 全員同じならこの直線
plt.xlabel('累積人口比'); plt.ylabel('累積所得比')
plt.title(f'ジニ係数 = {g:.3f}'); plt.legend(); plt.show()

> ⚠️ **よくある間違い**：サンプルを増やしても、**取り方が偏っていれば**誤差（偏り）は減りません。大事なのは数より**無作為性**（偏りのない抽出）です。

## 5. コレログラム（自己相関）— 時系列の季節性を見抜く

時系列が「何か月前の自分」と似ているかを測るのが**自己相関**、それを並べた図が**コレログラム**です。季節性のある月次データで描いてみます。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
t = np.arange(72)                          # 6年分の月次
y = 50 + 0.3*t + 10*np.sin(2*np.pi*t/12) + rng.normal(0, 2, 72)  # トレンド＋12か月周期＋雑音
from statsmodels.graphics.tsaplots import plot_acf
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(t, y); a1.set_title('月次データ（季節性あり）'); a1.set_xlabel('月'); a1.set_ylabel('値')
plot_acf(y, lags=30, ax=a2); a2.set_title('コレログラム（自己相関）')
plt.tight_layout(); plt.show()

> 💡 ラグ12・24で自己相関が高く出る＝**12か月周期の季節性**。点線（青い帯）の外にある棒が「無相関とは言えない（有意）」自己相関です。売上やアクセス数の季節パターンを見つけるのに使います。

## 6. ラスパイレス指数 — 価格の変化だけを取り出す

物価の指数は「価格も数量も動く」中で**価格変化だけ**を測りたい。**基準年の数量で重みづけ**するのがラスパイレス指数です（消費者物価指数もこの考え方）。

In [ ]:
import numpy as np, pandas as pd
tbl = pd.DataFrame({'品目':['米','パン','麺'],
                    '価格_基準年':[400, 250, 150], '数量_基準年':[10, 8, 12],
                    '価格_比較年':[440, 300, 150]})
las = (tbl['価格_比較年']*tbl['数量_基準年']).sum() / (tbl['価格_基準年']*tbl['数量_基準年']).sum() * 100
print(f'ラスパイレス価格指数 = {las:.1f}（基準年=100）→ 全体で約{las-100:.1f}%の値上がり')
tbl

> 💡 分母も分子も**数量は基準年で固定**。だから「数量の変化」に邪魔されず価格変化だけを比べられます。4級で扱った消費者物価指数(CPI)も、根っこはこのラスパイレス型です。

## 7. 切断効果（選抜バイアス）— 選んだ後に相関が消える

全体では相関があるのに、**合格者など一部だけを抜き出すと相関が弱く（時に逆に）見える**現象です。

In [ ]:
rng = np.random.default_rng(1)
x = rng.normal(50, 10, 1000)            # 例：筆記の力
y = 0.6*x + rng.normal(0, 8, 1000)      # 例：面接の力（全体では正の相関）
print('全体の相関       :', round(np.corrcoef(x, y)[0,1], 3))
sel = (x + y) > np.quantile(x + y, 0.7)  # 合計上位30%だけ「合格」
print('合格者だけの相関 :', round(np.corrcoef(x[sel], y[sel])[0,1], 3))
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(x[~sel], y[~sel], s=12, alpha=.3, label='不合格')
ax.scatter(x[sel],  y[sel],  s=12, alpha=.6, color='crimson', label='合格(上位30%)')
ax.set_xlabel('筆記'); ax.set_ylabel('面接'); ax.legend(); ax.set_title('切断効果：選抜後は相関が弱まる')
plt.show()

> ⚠️ 「選抜された集団」での相関は、母集団の相関より弱くなりがちです（range restriction）。「難関大の学生では入試成績と入学後成績が無相関」のような話は、たいていこの切断効果が原因です。
>
> 💡 採用・与信など**閾値で絞ったあとのデータ**で相関を見るときは、選抜バイアスを疑いましょう。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
from scipy import stats
# Q1: 左右対称データ[1,2,3,4,5]の歪度を ans に（ほぼ0）
ans = None   # 例: stats.skew([1,2,3,4,5])
_check('Q1 歪度', ans, stats.skew([1,2,3,4,5]))

In [ ]:
# Q2: 全員同じ所得[10,10,10,10]のジニ係数を ans に（完全平等）
ans = None   # 例: gini([10,10,10,10])[0]
_check('Q2 ジニ係数', ans, gini([10,10,10,10])[0])

In [ ]:
# Q3: 1000人中ある層が250人。層化抽出で全体100人取るときその層から何人？を ans に
ans = None   # 250/1000*100
_check('Q3 層の人数', ans, 250/1000*100)

In [ ]:
import numpy as np
# Q4: 基準年価格[100,200],数量[5,10], 比較年価格[120,210] のラスパイレス指数を ans に
p0 = np.array([100,200]); q0 = np.array([5,10]); pt = np.array([120,210])
ans = None   # 例: (pt*q0).sum()/(p0*q0).sum()*100
_check('Q4 ラスパイレス指数', ans, (pt*q0).sum()/(p0*q0).sum()*100)

---
## 🏆 練習問題

**問1.** 1000人の母集団から、層化抽出で各年代から**人数比に応じて**合計100人を抽出しよう
（ヒント：各層 `len(層)/1000*100` 人）。

**問2.** `students_scores.csv` の `数学`・`国語` の歪度を比べ、より左右対称なのはどちらか答えよう。

**問3.** 2つの社会 `A=[10,10,10,10,10]`（平等）と `B=[1,2,5,12,80]`（格差大）の
ジニ係数を計算し、値の大小と直感が合うか確かめよう。

**問4.** 切断効果のコードで、合格ラインを「上位30%」から「上位10%」に厳しくすると、合格者の相関はどう変わる？ `np.quantile(..., 0.9)` にして確かめよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


> 🔑 **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから 👉 **[07_抽出法・実験計画・格差指標 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/07_%E6%8A%BD%E5%87%BA%E6%B3%95%E3%83%BB%E5%AE%9F%E9%A8%93%E8%A8%88%E7%94%BB%E3%83%BB%E6%A0%BC%E5%B7%AE%E6%8C%87%E6%A8%99.md)**

🎉 これで**統計検定2級の主要範囲を大幅にカバー**しました。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 単純無作為/系統/層化 | 完全ランダム/一定間隔/層ごと |
| クラスター/多段 | 集団ごと/段階的 |
| フィッシャーの3原則 | 反復・無作為化・局所管理 |
| 歪度/尖度 | 非対称さ/とがり |
| ジニ係数 | 格差(0平等〜1独占) |

In [ ]:
# チートシート（抽出・格差）
import pandas as pd, numpy as np
from scipy import stats
pop = pd.DataFrame({'id':range(100), '層':np.random.default_rng(0).choice(['A','B'],100)})
pop.groupby('層', group_keys=False).sample(5, random_state=1)   # 層化抽出
print(stats.skew([1,2,3,10]), stats.kurtosis([1,2,3,10]))      # 歪度・尖度